<a href="https://colab.research.google.com/github/jennyk23/Magpy/blob/main/Untitled4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%bash
set -e

export DEBIAN_FRONTEND=noninteractive

apt-get update -qq
apt-get install -y mariadb-server mariadb-client > /dev/null

mkdir -p /run/mysqld
chown mysql:mysql /run/mysqld

service mariadb start > /dev/null 2>&1 || true

for i in {1..20}; do
    if mariadb-admin -u root ping --silent; then
        mariadb --version
        echo "MariaDB iniciado com sucesso."
        exit 0
    fi

    sleep 1
done

echo "Não foi possível iniciar o MariaDB."
exit 1

mysqld is alive
mariadb  Ver 15.1 Distrib 10.6.23-MariaDB, for debian-linux-gnu (x86_64) using  EditLine wrapper
MariaDB iniciado com sucesso.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
from pathlib import Path

sql = r"""
-- ============================================================
-- ATIVIDADE 1 – SISTEMA DE PEDIDOS
-- MYSQL/MARIADB
-- ============================================================

DROP DATABASE IF EXISTS empresa_vendas;

CREATE DATABASE empresa_vendas
    CHARACTER SET utf8mb4
    COLLATE utf8mb4_unicode_ci;

USE empresa_vendas;

-- ============================================================
-- TABELA CLIENTE
-- ============================================================

CREATE TABLE cliente (
    id_cliente INT AUTO_INCREMENT,
    nome VARCHAR(120) NOT NULL,
    email VARCHAR(150) NOT NULL,

    CONSTRAINT pk_cliente
        PRIMARY KEY (id_cliente),

    CONSTRAINT uq_cliente_email
        UNIQUE (email),

    CONSTRAINT chk_cliente_nome
        CHECK (
            CHAR_LENGTH(TRIM(nome)) > 0
        )
) ENGINE = InnoDB;

-- ============================================================
-- TABELA PRODUTO
-- ============================================================

CREATE TABLE produto (
    id_produto INT AUTO_INCREMENT,
    nome VARCHAR(120) NOT NULL,
    preco DECIMAL(10,2) NOT NULL,
    estoque INT NOT NULL DEFAULT 0,

    CONSTRAINT pk_produto
        PRIMARY KEY (id_produto),

    CONSTRAINT chk_produto_preco
        CHECK (preco > 0),

    CONSTRAINT chk_produto_estoque
        CHECK (estoque >= 0)
) ENGINE = InnoDB;

-- ============================================================
-- TABELA PEDIDO
-- ============================================================

CREATE TABLE pedido (
    id_pedido INT AUTO_INCREMENT,
    id_cliente INT NOT NULL,

    data_pedido DATETIME
        NOT NULL
        DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT pk_pedido
        PRIMARY KEY (id_pedido),

    CONSTRAINT fk_pedido_cliente
        FOREIGN KEY (id_cliente)
        REFERENCES cliente(id_cliente)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
) ENGINE = InnoDB;

-- ============================================================
-- TABELA ITEM_PEDIDO
-- ============================================================

CREATE TABLE item_pedido (
    id_pedido INT NOT NULL,
    id_produto INT NOT NULL,
    quantidade INT NOT NULL,

    CONSTRAINT pk_item_pedido
        PRIMARY KEY (
            id_pedido,
            id_produto
        ),

    CONSTRAINT chk_item_quantidade
        CHECK (quantidade > 0),

    CONSTRAINT fk_item_pedido_pedido
        FOREIGN KEY (id_pedido)
        REFERENCES pedido(id_pedido)
        ON UPDATE CASCADE
        ON DELETE CASCADE,

    CONSTRAINT fk_item_pedido_produto
        FOREIGN KEY (id_produto)
        REFERENCES produto(id_produto)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
) ENGINE = InnoDB;

-- ============================================================
-- TRIGGERS DE ESTOQUE
-- ============================================================

DROP TRIGGER IF EXISTS trg_validar_estoque_insert;
DROP TRIGGER IF EXISTS trg_atualizar_estoque_insert;

DELIMITER $$

-- Verifica se há estoque suficiente antes da venda
CREATE TRIGGER trg_validar_estoque_insert
BEFORE INSERT ON item_pedido
FOR EACH ROW
BEGIN
    DECLARE v_estoque INT;

    IF NEW.quantidade <= 0 THEN

        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT =
            'A quantidade deve ser maior que zero.';

    END IF;

    SELECT estoque
    INTO v_estoque
    FROM produto
    WHERE id_produto = NEW.id_produto;

    IF v_estoque < NEW.quantidade THEN

        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT =
            'Estoque insuficiente para realizar a venda.';

    END IF;
END$$

-- Atualiza o estoque após inserir um item
CREATE TRIGGER trg_atualizar_estoque_insert
AFTER INSERT ON item_pedido
FOR EACH ROW
BEGIN
    UPDATE produto
    SET estoque = estoque - NEW.quantidade
    WHERE id_produto = NEW.id_produto;
END$$

DELIMITER ;

-- ============================================================
-- INSERÇÃO DOS 2 CLIENTES
-- ============================================================

INSERT INTO cliente (
    nome,
    email
) VALUES
(
    'Ana Paula Martins',
    'ana.martins@email.com'
),
(
    'Carlos Henrique Souza',
    'carlos.souza@email.com'
);

-- ============================================================
-- INSERÇÃO DOS 3 PRODUTOS
-- ============================================================

INSERT INTO produto (
    nome,
    preco,
    estoque
) VALUES
(
    'Notebook',
    3500.00,
    20
),
(
    'Mouse sem fio',
    120.00,
    15
),
(
    'Teclado mecânico',
    280.00,
    10
);

-- ============================================================
-- INSERÇÃO DO PEDIDO PRINCIPAL
-- ============================================================

INSERT INTO pedido (
    id_cliente,
    data_pedido
) VALUES (
    1,
    CURRENT_TIMESTAMP
);

SET @id_pedido_principal = LAST_INSERT_ID();

-- ============================================================
-- INSERÇÃO DE 2 ITENS NO PEDIDO
-- ============================================================

INSERT INTO item_pedido (
    id_pedido,
    id_produto,
    quantidade
) VALUES
(
    @id_pedido_principal,
    1,
    2
),
(
    @id_pedido_principal,
    2,
    1
);

-- A inserção acima reduz automaticamente:
-- Notebook: 20 para 18
-- Mouse: 15 para 14

-- ============================================================
-- PEDIDO VAZIO PARA DEMONSTRAR A EXCLUSÃO
-- ============================================================

INSERT INTO pedido (
    id_cliente,
    data_pedido
) VALUES (
    2,
    CURRENT_TIMESTAMP
);

-- ============================================================
-- EXCLUIR PEDIDOS SEM ITENS
-- ============================================================

DELETE p
FROM pedido AS p

LEFT JOIN item_pedido AS ip
    ON ip.id_pedido = p.id_pedido

WHERE ip.id_pedido IS NULL;
"""

arquivo_sql = Path(
    "/content/atividade1_sistema_pedidos.sql"
)

arquivo_sql.write_text(
    sql,
    encoding="utf-8"
)

print("Script salvo em:", arquivo_sql)
print("Tamanho:", arquivo_sql.stat().st_size, "bytes")


Script salvo em: /content/atividade1_sistema_pedidos.sql
Tamanho: 5787 bytes


In [3]:
import subprocess
from pathlib import Path

arquivo_sql = Path(
    "/content/atividade1_sistema_pedidos.sql"
)

with arquivo_sql.open(
    "r",
    encoding="utf-8"
) as arquivo:

    processo = subprocess.run(
        [
            "mariadb",
            "-u",
            "root",
            "--default-character-set=utf8mb4"
        ],
        stdin=arquivo,
        text=True,
        capture_output=True
    )

print(processo.stdout)

if processo.returncode != 0:
    print("ERRO DO MYSQL/MARIADB:")
    print(processo.stderr)

    raise RuntimeError(
        "A execução foi interrompida."
    )

print(
    "Banco empresa_vendas criado com sucesso."
)


Banco empresa_vendas criado com sucesso.


In [4]:
import subprocess

consultas = r"""
USE empresa_vendas;

SELECT
    'CLIENTES CADASTRADOS' AS etapa;

SELECT
    id_cliente,
    nome,
    email
FROM cliente
ORDER BY id_cliente;

SELECT
    'PRODUTOS E ESTOQUE APÓS A VENDA' AS etapa;

SELECT
    id_produto,
    nome,
    preco,
    estoque
FROM produto
ORDER BY id_produto;

SELECT
    'ITENS DO PEDIDO' AS etapa;

SELECT
    p.id_pedido,
    c.nome AS cliente,
    pr.nome AS produto,
    pr.preco,
    ip.quantidade,

    ROUND(
        pr.preco * ip.quantidade,
        2
    ) AS subtotal

FROM pedido AS p

INNER JOIN cliente AS c
    ON c.id_cliente = p.id_cliente

INNER JOIN item_pedido AS ip
    ON ip.id_pedido = p.id_pedido

INNER JOIN produto AS pr
    ON pr.id_produto = ip.id_produto

ORDER BY
    p.id_pedido,
    pr.nome;

SELECT
    'VALOR TOTAL DO PEDIDO' AS etapa;

SELECT
    p.id_pedido,
    c.nome AS cliente,

    ROUND(
        SUM(
            ip.quantidade * pr.preco
        ),
        2
    ) AS valor_total_pedido

FROM pedido AS p

INNER JOIN cliente AS c
    ON c.id_cliente = p.id_cliente

INNER JOIN item_pedido AS ip
    ON ip.id_pedido = p.id_pedido

INNER JOIN produto AS pr
    ON pr.id_produto = ip.id_produto

GROUP BY
    p.id_pedido,
    c.id_cliente,
    c.nome

ORDER BY p.id_pedido;

SELECT
    'CONTAGEM FINAL' AS etapa;

SELECT
    'Clientes' AS tabela,
    COUNT(*) AS quantidade
FROM cliente

UNION ALL

SELECT
    'Produtos',
    COUNT(*)
FROM produto

UNION ALL

SELECT
    'Pedidos',
    COUNT(*)
FROM pedido

UNION ALL

SELECT
    'Itens do pedido',
    COUNT(*)
FROM item_pedido;

SELECT
    'TRIGGERS CRIADAS' AS etapa;

SHOW TRIGGERS
FROM empresa_vendas;
"""

resultado = subprocess.run(
    [
        "mariadb",
        "-u",
        "root",
        "--table",
        "--default-character-set=utf8mb4",
        "-e",
        consultas
    ],
    text=True,
    capture_output=True
)

print(resultado.stdout)

if resultado.returncode != 0:
    print("ERRO DO MYSQL/MARIADB:")
    print(resultado.stderr)

+----------------------+
| etapa                |
+----------------------+
| CLIENTES CADASTRADOS |
+----------------------+
+------------+-----------------------+------------------------+
| id_cliente | nome                  | email                  |
+------------+-----------------------+------------------------+
|          1 | Ana Paula Martins     | ana.martins@email.com  |
|          2 | Carlos Henrique Souza | carlos.souza@email.com |
+------------+-----------------------+------------------------+
+----------------------------------+
| etapa                            |
+----------------------------------+
| PRODUTOS E ESTOQUE APÓS A VENDA  |
+----------------------------------+
+------------+-------------------+---------+---------+
| id_produto | nome              | preco   | estoque |
+------------+-------------------+---------+---------+
|          1 | Notebook          | 3500.00 |      18 |
|          2 | Mouse sem fio     |  120.00 |      14 |
|          3 | Teclado mecânico

In [ ]:
import subprocess

sql_delete_query = r"""
USE empresa_vendas;
DELETE p
FROM pedido AS p
LEFT JOIN item_pedido AS ip
    ON ip.id_pedido = p.id_pedido
WHERE ip.id_pedido IS NULL;
"""

resultado = subprocess.run(
    [
        "mariadb",
        "-u",
        "root",
        "--default-character-set=utf8mb4",
        "-e",
        sql_delete_query
    ],
    text=True,
    capture_output=True
)

print(resultado.stdout)

if resultado.returncode != 0:
    print("ERRO DO MYSQL/MARIADB:")
    print(resultado.stderr)
    raise RuntimeError("A execução foi interrompida.")

print("Pedidos sem itens excluídos com sucesso.")